# Histogram binning (research prototype)

Count `n` values in `[0, 1)` into `NBINS` equal-width bins. The values use a fixed
deterministic pseudo-random fill so the result is reproducible; we report a weighted
checksum (sum over bins of `counts[b] * (b + 1)`) as the oracle -- the weight makes it
sensitive to WHICH bin each value landed in, not just the total. The C port must
implement the same binning and match this checksum, and its counts must sum to `n`.

In [ ]:
N_TEST = 1_000_000
NBINS = 256

# deterministic pseudo-random values in [0, 1): a 32-bit multiplicative hash of the
# index (exact 64-bit modular arithmetic, so C reproduces it bit for bit)
def x_of(i):
    return ((i * 2654435761 + 12345) % 4294967296) / 4294967296.0

x = [x_of(i) for i in range(N_TEST)]

In [ ]:
# bin the values: index = floor(v * nbins), clamped to the top bin
def histogram(x, nbins):
    counts = [0] * nbins
    for v in x:
        b = int(v * nbins)
        if b >= nbins:
            b = nbins - 1
        counts[b] += 1
    return counts

counts = histogram(x, NBINS)
checksum = sum(counts[b] * (b + 1) for b in range(NBINS))
print('total  =', sum(counts))     # -> 1000000
print('checksum =', checksum)      # -> 128499644

In [ ]:
# The oracle: this notebook's own output. A port of the algorithm (in any
# language) must reproduce this checksum within tol. The per-bin weighted counts
# are handed over as the terms of the checksum, so the extraction tool can measure
# how much the sum legitimately moves under reordering (here: not at all -- counts
# are exact integers, so every summation order gives the same double).
ORACLE = {'kernel': 'histogram',
          'params': {'N': N_TEST, 'NBINS': NBINS},
          'expected': {'CHECKSUM': float(checksum)},
          'terms': {'CHECKSUM': [float(counts[b] * (b + 1)) for b in range(NBINS)]},
          'tol': 0.5}